## Аналіз A/B-тестів

Ви - аналітик даних в ІТ-компанії і до вас надійшла задача проаналізувати дані A/B тесту в популярній [грі Cookie Cats](https://www.facebook.com/cookiecatsgame). Це - гра-головоломка в стилі «з’єднай три», де гравець повинен з’єднати плитки одного кольору, щоб очистити дошку та виграти рівень. На дошці також зображені співаючі котики :)

Під час проходження гри гравці стикаються з воротами, які змушують їх чекати деякий час, перш ніж вони зможуть прогресувати або зробити покупку в додатку.

У цьому блоці завдань ми проаналізуємо результати A/B тесту, коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40. Зокрема, ми хочемо зрозуміти, як це вплинуло на утримання (retention) гравців. Тобто хочемо зрозуміти, чи переміщення воріт на 10 рівнів пізніше якимось чином вплинуло на те, що користувачі перестають грати в гру раніше чи пізніше з точки зору кількості їх днів з моменту встановлення гри.

Будемо працювати з даними з файлу `cookie_cats.csv`. Колонки в даних наступні:

- `userid` - унікальний номер, який ідентифікує кожного гравця.
- `version` - чи потрапив гравець в контрольну групу (gate_30 - ворота на 30 рівні) чи тестову групу (gate_40 - ворота на 40 рівні).
- `sum_gamerounds` - кількість ігрових раундів, зіграних гравцем протягом першого тижня після встановлення
- `retention_1` - чи через 1 день після встановлення гравець повернувся і почав грати?
- `retention_7` - чи через 7 днів після встановлення гравець повернувся і почав грати?

Коли гравець встановлював гру, його випадковим чином призначали до групи gate_30 або gate_40.

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.stats.api as sms
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from math import ceil

%matplotlib inline

1. Для початку, уявімо, що ми тільки плануємо проведення зазначеного А/B-тесту і хочемо зрозуміти, дані про скількох користувачів нам треба зібрати, аби досягнути відчутного ефекту. Відчутним ефектом ми вважатимемо збільшення утримання на 1% після внесення зміни. Обчисліть, скільки користувачів сумарно нам треба аби досягнути такого ефекту, якщо продакт менеджер нам повідомив, що базове утримання є 19%.

In [4]:
effect_size = sms.proportion_effectsize(0.20, 0.19)

In [5]:
effect_size

np.float64(0.025241594409087353)

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90189 entries, 0 to 90188
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   userid          90189 non-null  int64 
 1   version         90189 non-null  object
 2   sum_gamerounds  90189 non-null  int64 
 3   retention_1     90189 non-null  bool  
 4   retention_7     90189 non-null  bool  
dtypes: bool(2), int64(2), object(1)
memory usage: 2.2+ MB


In [6]:
required_n = sms.NormalIndPower().solve_power(
    effect_size,
    power=0.8,
    alpha=0.05,
    ratio=1
    )

required_n = ceil(required_n)

print(required_n)

24638


Нам потрібно буде мінімум 24638 спостережень для кожної групи.

2. Зчитайте дані АВ тесту у змінну `df` та виведіть середнє значення показника показник `retention_7` (утримання на 7 день) по версіям гри. Сформулюйте гіпотезу: яка версія дає краще утримання через 7 днів після встановлення гри?

In [8]:
df = pd.read_csv('cookie_cats.csv')
df.head()

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


In [11]:
session_counts = df['userid'].value_counts(ascending=False)
multi_users = session_counts[session_counts > 1].count()

print(f'Є {multi_users} користувачів, які зустрічаються кілька разів у наборі даних.')

Є 0 користувачів, які зустрічаються кілька разів у наборі даних.


In [13]:
retention_by_version = df.groupby('version')['retention_7'].mean()
print('Середнє значення показника утримання на 7 день за цільовими групами')
print(retention_by_version)

Середнє значення показника утримання на 7 день за цільовими групами
version
gate_30    0.190201
gate_40    0.182000
Name: retention_7, dtype: float64


Гіпотеза
Можемо сказати, що перенесення воріт з 40 рівня на 30 - тримає увагу гравця краще.


𝐻0:𝑝=𝑝0 - базове утримання користувача 19%


𝐻𝑎:𝑝>𝑝0 - утримання користувача зросло, принаймні на 1 %, після внесених змін.




3. Перевірте з допомогою пасуючого варіанту z-тесту, чи дає якась з версій гри кращий показник `retention_7` на рівні значущості 0.05. Обчисліть також довірчі інтервали для варіантів до переміщення воріт і після. Виведіть результат у форматі:

    ```
    z statistic: ...
    p-value: ...
    Довірчий інтервал 95% для групи control: [..., ...]
    Довірчий інтервал 95% для групи treatment: [..., ...]
    ```

    де замість `...` - обчислені значення.
    
    В якості висновку дайте відповідь на два питання:  

      1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?   
      2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  


In [14]:
control_sample = df[df['version'] == 'gate_30'].sample(n=required_n, random_state=22)
treatment_sample = df[df['version'] == 'gate_40'].sample(n=required_n, random_state=22)

ab_test = pd.concat([control_sample, treatment_sample], axis=0)
ab_test.reset_index(drop=True, inplace=True)

In [15]:
ab_test.head()

,userid,version,sum_gamerounds,retention_1,retention_7
0,7540471,gate_30,45,True,False
1,3589138,gate_30,21,True,False
2,3177668,gate_30,14,True,False
3,2133884,gate_30,26,False,False
4,492763,gate_30,39,True,True


In [16]:
ab_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49276 entries, 0 to 49275
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   userid          49276 non-null  int64 
 1   version         49276 non-null  object
 2   sum_gamerounds  49276 non-null  int64 
 3   retention_1     49276 non-null  bool  
 4   retention_7     49276 non-null  bool  
dtypes: bool(2), int64(2), object(1)
memory usage: 1.2+ MB


In [17]:
ab_test['version'].value_counts()

,count
version,
gate_30,24638
gate_40,24638


In [25]:
from statsmodels.stats.proportion import proportions_ztest, proportion_confint


control_results = ab_test[ab_test['version'] == 'gate_30']['retention_7']
treatment_results = ab_test[ab_test['version'] == 'gate_40']['retention_7']


n_con = control_results.count()
n_treat = treatment_results.count()
successes = [control_results.sum(), treatment_results.sum()]
nobs = [n_con, n_treat]

z_stat, pval = proportions_ztest(successes, nobs=nobs)
(lower_con, lower_treat), (upper_con, upper_treat) = proportion_confint(successes, nobs=nobs, alpha=0.05)

print(f'z statistic: {z_stat:.2f}')
print(f'p-value: {pval:.3f}')
print(f'Довірчий інтервал 95% для групи control: [{lower_con:.3f}, {upper_con:.3f}]')
print(f'Довірчий інтервал 95% для групи treatment: [{lower_treat:.3f}, {upper_treat:.3f}]')


z statistic: 3.20
p-value: 0.001
Довірчий інтервал 95% для групи control: [0.184, 0.194]
Довірчий інтервал 95% для групи treatment: [0.173, 0.183]


Отже, p-value 0.001 , набагато менше за 𝛼  = 0.05, ми  відхиляємо нульову гіпотезу і стверджуємо, що  коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40 - це вплинуло на утримання (retention) гравців.


Довірчі інтервали для груп control [0.184, 0.194] та treatment [0.173, 0.183] не перетинаються, що свідчить про наявність статистично значущої різниці між групами. Це означає, що утримання для груп gate_30 буде завжи вище за gate_40.

4. Виконайте тест Хі-квадрат на рівні значущості 5% аби визначити, чи є залежність між версією гри та утриманням гравця на 7ий день після реєстрації.

    - Напишіть, як для цього тесту будуть сформульовані гіпотези.
    - Проведіть обчислення, виведіть p-значення і напишіть висновок за результатами тесту.


Гіпотези:

𝐻0 : залежність між версією гри та утриманням гравця на 7-ий день після реєстрації однакова.


𝐻𝑎 : є різниця між версією гри та утриманням користувача.

In [22]:
pd.crosstab(ab_test['version'], ab_test['retention_7'])

retention_7,False,True
version,,
gate_30,19978,4660
gate_40,20253,4385


In [23]:
crosstab = pd.crosstab(ab_test['version'], ab_test['retention_7'])

In [24]:
chi2, p, dof, expected = stats.chi2_contingency(crosstab)

print(f"χ² = {chi2:.3f}")
print(f"p-value = {p:.5f}")
print(f"Ступені свободи = {dof}")
print("Очікувані частоти:\n", expected)

χ² = 10.166
p-value = 0.00143
Ступені свободи = 1
Очікувані частоти:
 [[20115.5  4522.5]
 [20115.5  4522.5]]


p-value = 0.00143 < 𝛼 = 0.05 , це означаю, що ми відхиляємо нульову гіпотезу і стверджуємо, що зміна воріт з рівня 30 на 40 - вплинуло на утримання користувачів.